In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split
import sys
sys.path.append('..')  # чтобы найти модуль utils, если он на уровень выше
from utils.extract_features import extract_features

print("Импорты выполнены.")

Импорты выполнены.


In [6]:
def tpr_at_1pct_fpr(y_true, y_pred_proba):
    """
    Вычисляет долю правильно обнаруженных атак (TPR)
    при уровне ложных срабатываний (FPR) = 1%.
    """
    # Сортируем примеры по убыванию вероятности атаки
    sorted_indices = np.argsort(y_pred_proba)[::-1]
    sorted_true = y_true[sorted_indices]
    
    total_neg = np.sum(y_true == 0)  # сколько всего нормальных запросов
    if total_neg == 0:
        return 0.0  # если нормальных нет, то не считаем
    
    target_fp = int(0.01 * total_neg)  # сколько ложных срабатываний мы можем позволить (1% от нормальных)
    fp = 0
    tp = 0
    
    # Проходим по отсортированным примерам
    for label in sorted_true:
        if label == 0:
            fp += 1
        else:
            tp += 1
        if fp >= target_fp:
            break  # остановка, когда набрали 1% ложных срабатываний
    
    total_pos = np.sum(y_true == 1)
    if total_pos == 0:
        return 0.0
    return tp / total_pos  # доля пойманных атак

In [7]:
df = pd.read_csv("../data/processed/balanced_unified_dataset.csv")
print(f"Всего строк: {len(df)}")
print("Уникальные источники:", df['source'].unique())
print("Первые 5 строк:")
print(df.head())

Всего строк: 70032
Уникальные источники: ['fwaf' 'csic2010' 'vulnxss' 'kaggle' 'httpparams' 'payloadbox']
Первые 5 строк:
                                                text  label    source
0                                     /thesis_award/      0      fwaf
1  /index.php?op=viewalbum&albumid=-1/**/union/**...      0      fwaf
2                                         /93223505/      0      fwaf
3  /atutor/include/classes/module/module.class.ph...      0      fwaf
4  http://localhost:8080/tienda1/global/titulo.js...      1  csic2010


In [8]:
print("Начинаем извлечение признаков из текстов...")
features_list = []
total = len(df)
for i, text in enumerate(df['text']):
    if i % 1000 == 0:
        print(f"Обработано {i}/{total}")
    # extract_features возвращает список чисел (признаков)
    features_list.append(extract_features(text, False))

X_all = pd.DataFrame(features_list)
y_all = df['label'].values
sources = df['source'].values

print(f"Признаки извлечены. Форма X_all: {X_all.shape}")

Начинаем извлечение признаков из текстов...
Обработано 0/70032
Обработано 1000/70032
Обработано 2000/70032
Обработано 3000/70032
Обработано 4000/70032
Обработано 5000/70032
Обработано 6000/70032
Обработано 7000/70032
Обработано 8000/70032
Обработано 9000/70032
Обработано 10000/70032
Обработано 11000/70032
Обработано 12000/70032
Обработано 13000/70032
Обработано 14000/70032
Обработано 15000/70032
Обработано 16000/70032
Обработано 17000/70032
Обработано 18000/70032
Обработано 19000/70032
Обработано 20000/70032
Обработано 21000/70032
Обработано 22000/70032
Обработано 23000/70032
Обработано 24000/70032
Обработано 25000/70032
Обработано 26000/70032
Обработано 27000/70032
Обработано 28000/70032
Обработано 29000/70032
Обработано 30000/70032
Обработано 31000/70032
Обработано 32000/70032
Обработано 33000/70032
Обработано 34000/70032
Обработано 35000/70032
Обработано 36000/70032
Обработано 37000/70032
Обработано 38000/70032
Обработано 39000/70032
Обработано 40000/70032
Обработано 41000/70032
Обр

In [9]:
source_names = ['fwaf', 'csic2010', 'kaggle', 'vulnxss', 'httpparams']
results = {}  # словарь для хранения результатов по каждому источнику

for test_source in source_names:
    print(f"\n===== Тестируем на {test_source} =====")
    
    # Маски для train/test
    test_mask = (sources == test_source)
    train_mask = ~test_mask
    
    # Данные для обучения и теста
    X_train = X_all[train_mask].values
    X_test = X_all[test_mask].values
    y_train = y_all[train_mask]
    y_test = y_all[test_mask]
    
    print(f"  Обучающих примеров: {len(X_train)}, тестовых: {len(X_test)}")
    
    # Создаём и обучаем Random Forest
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'  # помогает при дисбалансе классов
    )
    rf.fit(X_train, y_train)
    
    # Предсказания вероятностей и бинарных меток
    y_proba = rf.predict_proba(X_test)[:, 1]   # вероятность класса 1 (атака)
    y_pred = (y_proba >= 0.5).astype(int)      # порог 0.5
    
    # Расчёт метрик
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    tpr = tpr_at_1pct_fpr(y_test, y_proba)
    
    # Сохраняем в словарь
    results[test_source] = {
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'f1': f1,
        'accuracy': acc,
        'tpr@1%fpr': tpr
    }
    
    # Печатаем краткий отчёт по текущему источнику
    print(f"  ROC-AUC: {roc_auc:.4f}, PR-AUC: {pr_auc:.4f}, F1: {f1:.4f}, Accuracy: {acc:.4f}, TPR@1%FPR: {tpr:.4f}")

print("\nЦикл завершён.")


===== Тестируем на fwaf =====
  Обучающих примеров: 35907, тестовых: 34125


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 0.0000, F1: 0.0000, Accuracy: 0.8263, TPR@1%FPR: 0.0000

===== Тестируем на csic2010 =====
  Обучающих примеров: 44873, тестовых: 25159


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 1.0000, F1: 0.0693, Accuracy: 0.0359, TPR@1%FPR: 0.0000

===== Тестируем на kaggle =====
  Обучающих примеров: 62689, тестовых: 7343
  ROC-AUC: 0.9810, PR-AUC: 0.9992, F1: 0.9925, Accuracy: 0.9852, TPR@1%FPR: 0.0001

===== Тестируем на vulnxss =====
  Обучающих примеров: 68504, тестовых: 1528


d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 1.0000, F1: 0.9710, Accuracy: 0.9437, TPR@1%FPR: 0.0000

===== Тестируем на httpparams =====
  Обучающих примеров: 69005, тестовых: 1027
  ROC-AUC: 0.9774, PR-AUC: 0.9667, F1: 0.5444, Accuracy: 0.6300, TPR@1%FPR: 0.9142

Цикл завершён.


In [10]:
# Преобразуем словарь в DataFrame, транспонируем, чтобы источники были строками
result_df = pd.DataFrame(results).T
print("\n=== Итоговая таблица LODO (Random Forest) ===")
print(result_df.round(4))

# Сохраняем в CSV
result_df.to_csv("lodo_rf.csv")
print("Результаты сохранены в lodo_rf.csv")


=== Итоговая таблица LODO (Random Forest) ===
            roc_auc  pr_auc      f1  accuracy  tpr@1%fpr
fwaf            NaN  0.0000  0.0000    0.8263     0.0000
csic2010        NaN  1.0000  0.0693    0.0359     0.0000
kaggle       0.9810  0.9992  0.9925    0.9852     0.0001
vulnxss         NaN  1.0000  0.9710    0.9437     0.0000
httpparams   0.9774  0.9667  0.5444    0.6300     0.9142
Результаты сохранены в lodo_rf.csv
